# Cvrie Unsupervised Learning

##### Fix project root for imports

In [ ]:
import sys
from pathlib import Path

projectRoot = Path.cwd().parent
if str(projectRoot) not in sys.path:
    sys.path.append(str(projectRoot))

## Data Pre-processing

### Data loading

In [ ]:
import csv
import pandas as pd

csvPath = projectRoot / "Dataset" / "Student_Dataset.csv"

dataParsed = []

# Open & Read the CSV
with open(csvPath, "r", encoding="utf-8-sig", newline="") as file:
    reader = csv.reader(file)

    for row in reader:
        row = [col.strip() for col in row]

        # 3 elements : ID, Color, Testimony
        if len(row) == 3:
            dataParsed.append({
                "ID": row[0],
                "Color": row[1],
                "Testimony": row[2]
            })

        # 2 elements : ID, Testimony
        elif len(row) == 2:
            dataParsed.append({
                "ID": row[0],
                "Color": "0x000000",
                "Testimony": row[1]
            })

df = pd.DataFrame(dataParsed)

if df.shape[0] == 0:
    print("No data parsed from the CSV file.", file=sys.stderr)
    exit(84)

df["ID"] = pd.to_numeric(df["ID"])

# "0x" -> "#"
df["PlotColor"] = df["Color"].apply(lambda c: str(c).replace("0x", "#"))

print(df.head())

### **NLP** (Natural Language Processing)

- Convert text to lowercase
- Remove punctuation
- Tokenization
- Remove stop words (common words such as “I”, “the”, etc.)
- Lemmatize words (reduce words to their root form. ex: “breathing” and “breathes” -> “breath”)

the bird Breathing ! -> the bird breathing ! -> the bird breathing -> bird breathing -> bird breath

In [ ]:
from Source.naturalLanguageProcessing import cleanTextNLTK, cleanTextST, cleanTextExpert

# NLTK NLP
df["TestimonyNLTK"] = df["Testimony"].apply(cleanTextNLTK)
# Expert NLP
df["TestimonyExpert"] = df["Testimony"].apply(cleanTextExpert)
# SetenceTransformer NLP
df["TestimonyST"] = df["Testimony"].apply(cleanTextST)

print("Before :")
print(df["Testimony"].iloc[0])
print()

print("After NLTK (Lemmatization) :")
print(df["TestimonyNLTK"].iloc[0])
print()

print("After Expert (Stemming) :")
print(df["TestimonyExpert"].iloc[0])
print()

print("After ST :")
print(df["TestimonyST"].iloc[0])

### **Vectorisation**

#### Term Frequency - Inverse Document Frequency (**TF-IDF**)
- **TF (Term Frequency):** Counts how many times a word appears in a testimony.
- **IDF (Inverse Document Frequency):** Reduces the weight of words that appear in many testimonies and increases the weight of rare.

##### **TF-IDF** **NLTK** with **Lemmatizing**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize vectorizer with a limit on the number of features (words) to keep
vectorizer = TfidfVectorizer(max_features=1000)

# Train the vectorizer on our cleaned testimonies and transform them into a matrix
X = vectorizer.fit_transform(df["TestimonyNLTK"])

print("Matrix size (Patients, Words) :", X.shape)

retainedWords = vectorizer.get_feature_names_out()
print("Retained words :")
print(retainedWords[:20])

**Problems :**

**ache, aching, achy -> same family**

Solution : Mapping of synonym and **Stemming** instead of Lemmatizing -> cut the end of the words to get the same root word

**afternoon -> useless**

Solution : **Custom stop words** -> to remove "afternoon", "morning", "also" ...

##### **TF-IDF** **Expert NLTK** with **Stemming** and **Custom Stop Words List**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizerExpert = TfidfVectorizer(max_features=1000, min_df=3, max_df=0.85)

XExpert = vectorizerExpert.fit_transform(df["TestimonyExpert"])

print("Matrix size (Patients, Words) :", XExpert.shape)

retainedWords = vectorizerExpert.get_feature_names_out()
print("Retained words :")
print(retainedWords[:30])

**Conclusion :**

The word matrix is **mostly composed** by **medical vocabulary** :
- **Anatomy** : "abdomen", "ankl", "blood", "bone", ...
- **Symptom** : "ach", "allergi", "anxieti", "vomit", ...
- **Pain indication** : "dull", "sharp", "tight", ...

Row (**Patients**) : 1011

Column (**Words**) : 716

#### **SetenceTransformer**

In [ ]:
from sentence_transformers import SentenceTransformer

modelST = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = modelST.encode(df["TestimonyST"].tolist(), show_progress_bar=True)

print(f"Embeddings shape: {embeddings.shape}")

## Clustering

### K-Means (**TF-IDF**)

#### Finding K (number of different clusters needed / number of diseases)

##### Data preparation for clustering

In [ ]:
from scipy.sparse import issparse, csr_matrix
import numpy as np
import umap

# Convert sparse matrix to dense if necessary
XDense = csr_matrix(XExpert).toarray() if issparse(XExpert) else np.array(XExpert)

# UMAP reduction to 42D for K-Means clustering
# We first was using PCA, but UMAP gives better results for clustering in this case
umap42D = umap.UMAP(n_components=42, random_state=42, metric='cosine', n_jobs=1)
Xumap42D = np.asarray(umap42D.fit_transform(XDense))

##### **Elbow Method**

Elbow Calculation

In [ ]:
from sklearn.cluster import KMeans

# Testing different K values to find the best balance
kRange = range(2, 31)
inertiaValuesKMTFIDF = []

# Fit K-Means for each K and store the inertia to analyze later
for k in kRange:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(Xumap42D)
    inertiaValuesKMTFIDF.append(km.inertia_)

Elbow Plotting

In [ ]:
import matplotlib.pyplot as plt

# Plotting the Elbow Method graph to visualize inertia values for different K
# and find the optimal number of clusters

fig, ax1 = plt.subplots(figsize=(8, 6))
fig.patch.set_facecolor('#0f0f1a')
ax1.set_facecolor('#1a1a2e')
ax1.tick_params(colors='#aaaacc')
ax1.xaxis.label.set_color('#aaaacc')
ax1.yaxis.label.set_color('#aaaacc')
ax1.title.set_color('#e0e0ff')
for spine in ax1.spines.values():
    spine.set_edgecolor('#333355')

ax1.plot(kRange, inertiaValuesKMTFIDF, color='#7b61ff', linewidth=2.5, marker='o',
         markerfacecolor='#ff6b9d', markersize=8, markeredgewidth=0)
ax1.set_title('Elbow Method', fontsize=14, fontweight='bold', pad=12)
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Inertia (intra-cluster distance)')
ax1.set_xticks(list(kRange))
ax1.tick_params(axis='x', rotation=45)
ax1.grid(True, color='#2a2a4a', linestyle='--', linewidth=0.8)

plt.tight_layout()
plt.show()

##### **Silhouette Score**

Silhouette Calculation

In [ ]:
from sklearn.metrics import silhouette_score
import numpy as np

silhouetteScoreKMTFIDF = []

# Fit K-Means for each K, predict labels, and compute silhouette scores to find the best K
for k in kRange:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(Xumap42D)
    silhouetteScoreKMTFIDF.append(silhouette_score(Xumap42D, labels))

bestKTFIDF = list(kRange)[np.argmax(silhouetteScoreKMTFIDF)]
print(f"Best K (TFIDF) : {bestKTFIDF} (silhouette score : {max(silhouetteScoreKMTFIDF):.4f})")

Silhouette Plotting

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Plotting the Silhouette Scores for different K values
# to visualize which K gives the best clustering quality

fig, ax2 = plt.subplots(figsize=(8, 6))
fig.patch.set_facecolor('#0f0f1a')
ax2.set_facecolor('#1a1a2e')
ax2.tick_params(colors='#aaaacc')
ax2.xaxis.label.set_color('#aaaacc')
ax2.yaxis.label.set_color('#aaaacc')
ax2.title.set_color('#e0e0ff')
for spine in ax2.spines.values():
    spine.set_edgecolor('#333355')

colors = ['#444466'] * len(silhouetteScoreKMTFIDF)
colors[np.argmax(silhouetteScoreKMTFIDF)] = '#ff6b9d'

ax2.bar(list(kRange), silhouetteScoreKMTFIDF, color=colors, edgecolor='none', width=0.6)
ax2.plot(kRange, silhouetteScoreKMTFIDF, color='#7b61ff', linewidth=2, linestyle='--',
         marker='o', markersize=6, markerfacecolor='#7b61ff', markeredgewidth=0)
ax2.axvline(x=bestKTFIDF, color='#ff6b9d', linestyle=':', linewidth=1.5, alpha=0.7)
ax2.set_title('Silhouette Score', fontsize=14, fontweight='bold', pad=12)
ax2.set_xlabel('Nombre de clusters (k)')
ax2.set_ylabel('Score silhouette')
ax2.set_xticks(list(kRange))
ax2.tick_params(axis='x', rotation=45)
ax2.grid(True, color='#2a2a4a', linestyle='--', linewidth=0.8, axis='y')
ax2.annotate(f'  best k={bestKTFIDF}',
             xy=(bestKTFIDF, silhouetteScoreKMTFIDF[bestKTFIDF - 2]),
             color='#ff6b9d', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

##### **Analyzing Results**

The inertia curve (Elbow Method) does **not show a clear elbow**, indicating the absence of naturally defined cluster structures in the data.

The maximum silhouette score is **~0.11 (k=18)**, which is poor (a good clustering typically exceeds 0.4). The scores remain low and stable, indicating low-cohesion clusters.

> **Limitations of TF-IDF:** Word frequency representation ignores meaning and context. This makes it difficult to cluster testimonies that have varied vocabulary but similar semantics.
> <br> For example:
>
> | Sentence | TF-IDF Tokens |
> |--------|---------------|
> | *"I can't sleep at night"* | `{sleep, night}` |
> | *"I suffer from insomnia"* | `{suffer, insomnia}` |
>
> These two sentences share no common words — TF-IDF separates them, even though they describe the exact same symptom.

This interpretation strongly encourages the use of **Sentence Transformers** (Deep Learning Embeddings) in the next section to capture context.

#### Training and evaluation

In [ ]:
import numpy as np
import umap
from scipy.sparse import csr_matrix
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

# K-Means TFIDF with the best K found (7 clusters)
kMeansTFIDF = KMeans(n_clusters=bestKTFIDF, random_state=42, n_init=10)
kMeansTFIDF.fit_predict(XExpert)

# UMAP (716 columns -> 2 columns : X & Y)
umapReducer = umap.UMAP(n_components=2, random_state=42, metric='cosine', n_jobs=1)
XExpertReduced = np.asarray(umapReducer.fit_transform(XExpert))

# Save 2d XY to DataFrame
umapX = XExpertReduced[:, 0]
umapY = XExpertReduced[:, 1]

#### Visualisation

In [ ]:
# Charts
fig, axes = plt.subplots(1, 2, figsize=(16, 7), facecolor='#0f0f1a')

# Plot 1: K-Means Prediction
ax1 = axes[0]
ax1.set_facecolor('#1a1a2e')
ax1.scatter(umapX, umapY, c=kMeansTFIDF.labels_, cmap='tab10', s=35, alpha=0.9)
ax1.set_title("1. K-Means TF-IDF Prediction (k=7)", color='#e0e0ff', fontsize=13, fontweight='bold')
ax1.tick_params(colors='#aaaacc')

# Plot 2: Dataset True Colors
ax2 = axes[1]
ax2.set_facecolor('#1a1a2e')
ax2.scatter(umapX, umapY, c=df["PlotColor"], s=35, alpha=0.9, edgecolors='w', linewidths=0.2)
ax2.set_title("2. Dataset True Colors", color='#e0e0ff', fontsize=13, fontweight='bold')
ax2.tick_params(colors='#aaaacc')

# General aesthetics
for ax in axes:
    ax.set_xlabel("UMAP Dimension 1", color='#aaaacc')
    ax.set_ylabel("UMAP Dimension 2", color='#aaaacc')
    for spine in ax.spines.values():
        spine.set_edgecolor('#333355')
    ax.grid(True, color='#2a2a4a', linestyle='--', linewidth=0.8, alpha=0.5)

fig.suptitle("K-Means TF-IDF (Prediction vs True Colors)", fontsize=15, fontweight='bold', color='#e0e0ff', y=1.02)
plt.tight_layout()
plt.show()

#### Cluster conclusion

In [ ]:
# Get the TF-IDF vocabulary (700+ terms) learned by the vectorizer
voc = vectorizerExpert.get_feature_names_out()

# Get the 7 K-Means cluster centroids
# Each centroid stores the average weight of every term for that cluster
clustersCenters = kMeansTFIDF.cluster_centers_

print("CLUSTER (DISEASE) ANALYSIS :")
print("Top keywords for each patient group:\n")

for i in range(len(clustersCenters)):
    # 5 highest-weighted terms in this cluster
    topIndex = clustersCenters[i].argsort()[::-1][:5]

    topWords = [str(voc[index]) for index in topIndex]

    print(f"#{i} -> Keywords: {", ".join(topWords)}")

### K-Means (**SetenceTransformers**)

#### Finding K (number of different clusters needed / number of diseases)

##### Data preparation for clustering

In [ ]:
import numpy as np
import umap

# UMAP reduction to 10D for K-Means clustering on ST embeddings
reducerKM = umap.UMAP(n_components=10, metric='cosine', random_state=42, n_jobs=1)
reducedKM = np.asarray(reducerKM.fit_transform(embeddings))

##### **Elbow Method**

Elbow Calculation

In [ ]:
from sklearn.cluster import KMeans

inertias = []
K = range(2, 20)

# Fit K-Means for each K and store the inertia to analyze later
for k in K:
    km = KMeans(n_clusters=k, random_state=42)
    km.fit(reducedKM)
    inertias.append(km.inertia_)

Elbow Plotting

In [ ]:
# Plotting the Elbow Method graph to visualize inertia values for different K
# and find the optimal number of clusters

fig, ax1 = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor('#0f0f1a')
ax1.set_facecolor('#1a1a2e')
ax1.tick_params(colors='#aaaacc')
ax1.xaxis.label.set_color('#aaaacc')
ax1.yaxis.label.set_color('#aaaacc')
ax1.title.set_color('#e0e0ff')
for spine in ax1.spines.values():
    spine.set_edgecolor('#333355')

ax1.plot(K, inertias, color='#7b61ff', linewidth=2.5, marker='o',
         markerfacecolor='#ff6b9d', markersize=8, markeredgewidth=0)
ax1.set_title('Elbow Method', fontsize=14, fontweight='bold', pad=12)
ax1.set_xlabel('Number of clusters (k)')
ax1.set_ylabel('Inertia')
ax1.set_xticks(list(K))
ax1.grid(True, color='#2a2a4a', linestyle='--', linewidth=0.8)

plt.tight_layout()
plt.show()

##### **Silhouette Score**

Silhouette Calculation

In [ ]:
silhouetteScoreKMST = []

# Fit K-Means for each K, predict labels, and compute silhouette scores to find the best K
for k in K:
    km = KMeans(n_clusters=k, random_state=42)
    labels = km.fit_predict(reducedKM)
    silhouetteScoreKMST.append(silhouette_score(reducedKM, labels))

bestKST = list(K)[np.argmax(silhouetteScoreKMST)]
print(f"Best K (ST) : {bestKST} (silhouette score : {max(silhouetteScoreKMST):.4f})")

Silhouette Plotting

In [ ]:
# Plotting the Silhouette Scores for different K values
# to visualize which K gives the best clustering quality

fig, ax2 = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor('#0f0f1a')
ax2.set_facecolor('#1a1a2e')
ax2.tick_params(colors='#aaaacc')
ax2.xaxis.label.set_color('#aaaacc')
ax2.yaxis.label.set_color('#aaaacc')
ax2.title.set_color('#e0e0ff')
for spine in ax2.spines.values():
    spine.set_edgecolor('#333355')

colors = ['#444466'] * len(silhouetteScoreKMST)
colors[np.argmax(silhouetteScoreKMST)] = '#ff6b9d'

ax2.bar(list(K), silhouetteScoreKMST, color=colors, edgecolor='none', width=0.6)
ax2.plot(K, silhouetteScoreKMST, color='#7b61ff', linewidth=2, linestyle='--',
         marker='o', markersize=6, markerfacecolor='#7b61ff', markeredgewidth=0)
ax2.axvline(x=bestKST, color='#ff6b9d', linestyle=':', linewidth=1.5, alpha=0.7)
ax2.set_title('Silhouette Score', fontsize=14, fontweight='bold', pad=12)
ax2.set_xlabel('Number of clusters (k)')
ax2.set_ylabel('Silhouette Score')
ax2.set_xticks(list(K))
ax2.grid(True, color='#2a2a4a', linestyle='--', linewidth=0.8, axis='y')
ax2.annotate(f'  best k={bestKST}',
             xy=(bestKST, silhouetteScoreKMST[bestKST - 2]),
             color='#ff6b9d', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

##### **Analyzing Results**

The inertia curve shows a **visible elbow around 5 ≤ k ≥ 8**

The maximum silhouette score is **0.64 (k=8)**, which is satisfactory
(above 0.5). The scores remain high between k=2 and k=10 before
gradually decreasing, indicating more cohesive and better-separated clusters.

> **Contribution of Sentence Transformers**: unlike TF-IDF, embeddings
> capture the meaning of sentences, which explains the significantly
> higher silhouette scores (~0.64 vs ~0.11).

Both methods converge towards **k=8** as the optimal value, offering a good
balance between intra-cluster cohesion and inter-cluster separation.

#### Training and evaluation

In [ ]:
import numpy as np
import umap
from sklearn.cluster import KMeans

# K-Means ST with the best K found
kMeansST = KMeans(n_clusters=bestKST, random_state=42, n_init=10)
kMeansST.fit_predict(reducedKM)

# UMAP (reduction to 2 columns : X & Y)
umapReducerST = umap.UMAP(n_components=2, random_state=42, metric='cosine', n_jobs=1)
embeddingsReduced = np.asarray(umapReducerST.fit_transform(embeddings))

# Save 2d XY to DataFrame
umapXST = embeddingsReduced[:, 0]
umapYST = embeddingsReduced[:, 1]

#### Visualisation

In [ ]:
import matplotlib.pyplot as plt

# Charts
fig, axes = plt.subplots(1, 2, figsize=(16, 7), facecolor='#0f0f1a')

# Plot 1: K-Means Prediction
ax1 = axes[0]
ax1.set_facecolor('#1a1a2e')
ax1.scatter(umapXST, umapYST, c=kMeansST.labels_, cmap='tab10', s=35, alpha=0.9)
ax1.set_title(f"1. K-Means ST Prediction (k={bestKST})", color='#e0e0ff', fontsize=13, fontweight='bold')
ax1.tick_params(colors='#aaaacc')

# Plot 2: Dataset True Colors
ax2 = axes[1]
ax2.set_facecolor('#1a1a2e')
ax2.scatter(umapXST, umapYST, c=df["PlotColor"], s=35, alpha=0.9, edgecolors='w', linewidths=0.2)
ax2.set_title("2. Dataset True Colors", color='#e0e0ff', fontsize=13, fontweight='bold')
ax2.tick_params(colors='#aaaacc')

# General aesthetics
for ax in axes:
    ax.set_xlabel("UMAP Dimension 1", color='#aaaacc')
    ax.set_ylabel("UMAP Dimension 2", color='#aaaacc')
    for spine in ax.spines.values():
        spine.set_edgecolor('#333355')
    ax.grid(True, color='#2a2a4a', linestyle='--', linewidth=0.8, alpha=0.5)

fig.suptitle("K-Means ST (Prediction vs True Colors)", fontsize=15, fontweight='bold', color='#e0e0ff', y=1.02)
plt.tight_layout()
plt.show()

#### Cluster conclusion

In [ ]:
from sklearn.metrics import pairwise_distances
import numpy as np

# Note: SentenceTransformers embeddings do not map directly to distinct words
# like TF-IDF does. Therefore, we cannot extract "top words" directly from centroids.
# Instead, we look at the testimonies closest to each centroid to understand the cluster.

print("CLUSTER (DISEASE) ANALYSIS :")
print("Top representative testimonies for each patient group:\n")

for i in range(bestKST):
    # Find the distance of all points in this cluster to the centroid
    clusterPointsIdx = np.where(kMeansST.labels_ == i)[0]
    clusterEmbeddings = reducedKM[clusterPointsIdx]
    centroid = kMeansST.cluster_centers_[i].reshape(1, -1)

    distances = pairwise_distances(clusterEmbeddings, centroid).flatten()

    # Get the indices of the 3 closest testimonies
    closestIndicesIdx = distances.argsort()[:3]
    closestIndices = clusterPointsIdx[closestIndicesIdx]

    testimonies = [df['Testimony'].iloc[idx][:50] + "..." for idx in closestIndices]

    print(f"#{i} -> Testimonies: {' | '.join(testimonies)}")

### HDBSCAN

**Why use HDBSCAN?**

While K-Means is efficient, it forces **every** testimony into a cluster. In a medical context, if a patient provides an off-topic or extremely vague testimony, K-Means will still assign it to a disease cluster, which is highly dangerous for diagnosis.

**HDBSCAN** (Hierarchical Density-Based Spatial Clustering of Applications with Noise) solves this issue by clustering based on data density. It groups highly similar testimonies together, but if a testimony is too isolated or vague, it is classified as **"Noise" / Outlier (Label -1)**. 

#### Testing HDBSCAN parameters

In [ ]:
from sklearn.cluster import HDBSCAN

print("Testing HDBSCAN parameters...")

# UMAP reduction to 10D for HDBSCAN clustering
reducerHDBScan = umap.UMAP(n_components=10, metric='cosine', random_state=42, n_jobs=1)
reducedHDBScan = np.asarray(reducerHDBScan.fit_transform(embeddings))

# Testing different minimum cluster sizes to find the best balance
for minCluster in [10, 15, 20, 25]:
    testCluster = HDBSCAN(min_cluster_size=minCluster, min_samples=10, copy=True)
    testLabels = testCluster.fit_predict(reducedHDBScan)

    nClusters = len(set(testLabels)) - (1 if -1 in testLabels else 0)
    nOutliers = list(testLabels).count(-1)

    print(f"min_cluster_size={minCluster} -> Found {nClusters} clusters, with {nOutliers} outliers")

# NOTE: Based on the output above, choose the min_cluster_size that gives a logical
# number of diseases (clusters) without classifying too many patients as outliers.

#### Training and evaluation

In [ ]:
from sklearn.cluster import HDBSCAN

# Training with logical parameters (min_samples=10)
finalClusterer = HDBSCAN(min_cluster_size=15, min_samples=10, copy=True)
df["ClusterHDBSCAN"] = finalClusterer.fit_predict(reducedHDBScan)

nClusters = df["ClusterHDBSCAN"].nunique() - (1 if -1 in df["ClusterHDBSCAN"].values else 0)
nOutliers = (df["ClusterHDBSCAN"] == -1).sum()

print(f"Final HDBSCAN Model: Found {nClusters} clusters and {nOutliers} outliers.")

#### Visualisation

In [ ]:
# UMAP reduction to 2D for visualization
coords2D = np.asarray(umap.UMAP(n_components=2, random_state=42, n_jobs=1).fit_transform(embeddings))

fig, axes = plt.subplots(1, 2, figsize=(16, 7), facecolor='#0f0f1a')

# Plot 1: HDBSCAN Clusters with Outliers
ax1 = axes[0]
ax1.set_facecolor('#1a1a2e')
maskOutliers = df["ClusterHDBSCAN"] == -1
maskClusters = df["ClusterHDBSCAN"] != -1

# Outliers -> dark gray
ax1.scatter(coords2D[maskOutliers, 0], coords2D[maskOutliers, 1], c='#444455', s=20, alpha=0.6, label="Outliers (Noise)")
# Draw valid clusters with their predicted colors
scatter1 = ax1.scatter(coords2D[maskClusters, 0], coords2D[maskClusters, 1], c=df["ClusterHDBSCAN"][maskClusters], cmap='tab20', s=35, alpha=0.9)

ax1.set_title(f"1. HDBSCAN Prediction ({nClusters} clusters)", color='#e0e0ff', fontsize=13, fontweight='bold')
ax1.legend(facecolor='#1a1a2e', edgecolor='#333355', labelcolor='#aaaacc')
ax1.tick_params(colors='#aaaacc')

# Plot 2: Dataset True Colors
ax2 = axes[1]
ax2.set_facecolor('#1a1a2e')
ax2.scatter(coords2D[:, 0], coords2D[:, 1], c=df["PlotColor"], s=35, alpha=0.9, edgecolors='w', linewidths=0.2)
ax2.set_title("2. Dataset True Colors (0x000000)", color='#e0e0ff', fontsize=13, fontweight='bold')
ax2.tick_params(colors='#aaaacc')

# General aesthetics
for ax in axes:
    ax.set_xlabel("UMAP Dimension 1", color='#aaaacc')
    ax.set_ylabel("UMAP Dimension 2", color='#aaaacc')
    for spine in ax.spines.values():
        spine.set_edgecolor('#333355')
    ax.grid(True, color='#2a2a4a', linestyle='--', linewidth=0.8, alpha=0.5)

fig.suptitle("HDBSCAN Outlier Detection vs True Colors", fontsize=15, fontweight='bold', color='#e0e0ff', y=1.02)
plt.tight_layout()
plt.show()

## Conclusion

<table>
  <thead>
    <tr>
      <th>NLP Method</th>
      <th>Clustering Algorithm</th>
      <th>Pros</th>
      <th>Cons</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><strong>TF-IDF</strong></td>
      <td>K-Means</td>
      <td>Very fast to compute.</td>
      <td>Fails completely due to lack of contextual understanding (Score ~0.11).</td>
    </tr>
    <tr>
      <td><strong>SentenceTransformers</strong></td>
      <td>K-Means</td>
      <td>Excellent contextual clustering, high silhouette score (~0.64).</td>
      <td>Forces every patient into a disease category, even irrelevant testimonies.</td>
    </tr>
    <tr>
      <td><strong>SentenceTransformers</strong></td>
      <td>HDBSCAN</td>
      <td>Detects invalid/vague testimonies (Outliers) based on density.</td>
      <td>Harder to tune hyperparameters perfectly.</td>
    </tr>
  </tbody>
</table>

### **Final Choice**
There is two possibilities for the finale choice :

**Every patient has a disease :**

In this case, the best model would be the K-Means with SentenceTransformers, he has very good silhouette score, excellent contextual clustering thanks to the SentenceTransformers, and forces every one into a cluster (disease category), there is no outliers for him.

**Not every patient has a disease (Possible Outliers) :**

In this second case, the best choice is HDBSCAN with SentenceTransformers, SentenceTransformers to keep this excellent contextual clustering, and HDBSCAN for the possibilities to detect outliers based on density.